# Tech Challenge - Fase 2
## Baseline da Fase 1

Este notebook reproduz o modelo final da Fase 1 (Regressão Logística otimizada com GridSearchCV), utilizando a mesma limpeza, a mesma divisão treino/teste e a mesma grade de hiperparâmetros.

O resultado é salvo em `results/baseline_fase1.json` e serve como referência para comparar o modelo otimizado pelo Algoritmo Genético no notebook 02.

In [1]:
from pathlib import Path

import json
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


## Preparação dos dados

As colunas `id` e `Unnamed: 32` são removidas e a variável alvo é convertida para `0` (Benigno) e `1` (Maligno).

In [2]:
caminho_dados = Path("/app/data/data.csv")

dados = pd.read_csv(caminho_dados)

df = dados.copy()

df = df.drop(
    columns=[
        "id",
        "Unnamed: 32"
    ],
    errors="ignore"
)

mapa_diagnostico = {
    "B": 0,
    "M": 1
}

df["diagnosis"] = df["diagnosis"].map(
    mapa_diagnostico
)

df["diagnosis"] = df["diagnosis"].astype(int)

df = df.drop_duplicates().reset_index(drop=True)

print(f"Dimensões finais: {df.shape}")
print()
print(df["diagnosis"].value_counts())

Dimensões finais: (569, 31)

diagnosis
0    357
1    212
Name: count, dtype: int64


## Divisão treino/teste

Divisão estratificada 80/20 com `random_state=42`, a mesma utilizada na Fase 1 e nos demais notebooks da Fase 2.

In [3]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Treino: {len(X_train)} registros")
print(f"Teste: {len(X_test)} registros")

Treino: 455 registros
Teste: 114 registros


## Modelo da Fase 1

Na Fase 1, a Regressão Logística foi selecionada como modelo final e otimizada com GridSearchCV, utilizando validação cruzada estratificada com 5 folds e Recall como critério de seleção.

In [ ]:

from sklearn.model_selection import StratifiedKFold, GridSearchCV

SEMENTE = 42

validacao_cruzada = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEMENTE
)

pipeline_fase1 = Pipeline([
    ("padronizacao", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=5000, random_state=SEMENTE))
])

grade_fase1 = {
    "modelo__C": [0.01, 0.1, 1, 10, 100],
    "modelo__class_weight": [None, "balanced"]
}

busca_fase1 = GridSearchCV(
    estimator=pipeline_fase1,
    param_grid=grade_fase1,
    scoring="recall",
    cv=validacao_cruzada,
    n_jobs=1,
    refit=True
)

busca_fase1.fit(X_train, y_train)
modelo_fase1 = busca_fase1.best_estimator_

print("Melhores hiperparâmetros (Fase 1):", busca_fase1.best_params_)
print(f"Recall médio CV: {busca_fase1.best_score_:.4f}")

previsoes_fase1 = modelo_fase1.predict(X_test)
probabilidades_fase1 = modelo_fase1.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, previsoes_fase1).ravel()

baseline_fase1 = {
    "modelo": "Regressão Logística - GridSearchCV (Fase 1)",
    "hiperparametros": {
        chave.replace("modelo__", ""): valor
        for chave, valor in busca_fase1.best_params_.items()
    },
    "recall_cv": float(busca_fase1.best_score_),
    "accuracy": float(accuracy_score(y_test, previsoes_fase1)),
    "precision": float(precision_score(y_test, previsoes_fase1)),
    "recall": float(recall_score(y_test, previsoes_fase1)),
    "f1": float(f1_score(y_test, previsoes_fase1)),
    "roc_auc": float(roc_auc_score(y_test, probabilidades_fase1)),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp)
}

print("\n=== RESULTADO FASE 1 (TESTE) ===")
for nome, valor in baseline_fase1.items():
    if isinstance(valor, float):
        print(f"{nome}: {valor:.4f}")
    else:
        print(f"{nome}: {valor}")

pasta_resultados = Path("/app/results")
pasta_resultados.mkdir(parents=True, exist_ok=True)

with open(pasta_resultados / "baseline_fase1.json", "w", encoding="utf-8") as arquivo:
    json.dump(baseline_fase1, arquivo, indent=4, ensure_ascii=False)

print("\nBaseline salvo em /app/results/baseline_fase1.json")

Melhores hiperparâmetros (Fase 1): {'modelo__C': 1, 'modelo__class_weight': 'balanced'}
Recall médio CV: 0.9588

=== RESULTADO FASE 1 (TESTE) ===
modelo: Regressão Logística - GridSearchCV (Fase 1)
hiperparametros: {'C': 1, 'class_weight': 'balanced'}
recall_cv: 0.9588
accuracy: 0.9737
precision: 0.9756
recall: 0.9524
f1: 0.9639
roc_auc: 0.9954
tn: 71
fp: 1
fn: 2
tp: 40

Baseline salvo em /app/results/baseline_fase1.json


## Conclusão

O modelo da Fase 1 (C = 1, class_weight = balanced) obteve no conjunto de teste Recall de 0,9524, F1 de 0,9639 e 2 falsos negativos.

Esses valores são a linha de base para avaliar o ganho obtido com o Algoritmo Genético no notebook 02.